## Download Dataset

In [ ]:
import gdown

# 2. Extract your File ID from your public Drive link:
# Format: https://drive.google.com/file/d/<FILE_ID>/view?usp=sharing
file_id = "1hEk9EmMmLzURNkpYFDi0uDC7YDYjbDG0"
url = f"https://drive.google.com/uc?id={file_id}"
output_path = "./data.zip"

# 3. Download the file
gdown.download(url, output_path, quiet=False)

In [ ]:
# -q: quiet (suppresses printing thousands of file paths)
# -o: overwrite existing files without prompting
!unzip -q -o ./data.zip -d ./

# Remove the original zip to free up disk space
!rm ./data.zip

## Download Sample Weights

In [ ]:
import gdown

# 2. Extract your File ID from your public Drive link:
# Format: https://drive.google.com/file/d/<FILE_ID>/view?usp=sharing
file_id = "1OIFa4WG6fTU9VPXY2iigMNV_c0XQaKeJ"
url = f"https://drive.google.com/uc?id={file_id}"
output_path = "./data/sample_weights.npy"

# 3. Download the file
gdown.download(url, output_path, quiet=False)

## Import

In [ ]:
!pip install torchvision albumentations pillow torchinfo matplotlib seaborn graphviz segmentation-models-pytorch jaxtyping beartype tqdm

In [ ]:
import gc
import glob
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torchvision
import torch.optim.lr_scheduler as lr_scheduler

from torch import Tensor, nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchinfo import summary

import albumentations as A
from albumentations.pytorch import ToTensorV2

from PIL import Image
from tqdm import tqdm
from typing import Dict, List, cast

from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp

from jaxtyping import Float, Int64, UInt8

## Configurations

In [ ]:
class Configuration:
    """Global configuration settings for data loading, training, and validation."""

    # Set to True when debugging locally to validate tensor layouts and dtypes.
    # Set to False on Kaggle/remote training to prevent beartype/jaxtyping dynamic
    # wrapper allocations from leaking host CPU RAM across long training sessions.
    SHOULD_CHECK: bool = False

    DEVICE: torch.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    NUM_DEVICES: int = torch.cuda.device_count()
    # Kaggle provides 2 vCPUs; 2 background workers efficiently overlap disk I/O with GPU execution
    NUM_WORKERS: int = 2

    NUM_CLASSES: int = 20
    EPOCHS: int = 20
    # Total batch size across both GPUs (8 per T4 GPU maintains stable throughput without VRAM OOM)
    BATCH_SIZE: int = 16 if torch.cuda.device_count() >= 2 else 8
    ACCUMULATION_STEPS: int = 1
    LR: float = 1e-4
    PATIENCE: int = 8

    APPLY_SHUFFLE: bool = True
    SEED: int = 768

    # Spatial dimensions of raw BDD100k images before preprocessing
    ORIGINAL_IMAGE_HEIGHT: int = 720
    ORIGINAL_IMAGE_WIDTH: int = 1280

    # Downsampled resolution chosen to balance spatial detail against GPU memory consumption
    IMAGE_HEIGHT: int = 360
    IMAGE_WIDTH: int = 640
    CHANNELS: int = 3  # Standard RGB channels


class Path:
    """Filesystem path definitions for datasets, masks, and precomputed weights."""

    BASE: str = "./data/bdd100k"

    SEGMENTATION_MASK_LABEL_FOLDER: str = BASE + "/segmentation_maps/color_labels"
    SEGMENTATION_MASK_TRAIN_PATH: str = SEGMENTATION_MASK_LABEL_FOLDER + "/train"
    SEGMENTATION_MASK_VAL_PATH: str = SEGMENTATION_MASK_LABEL_FOLDER + "/val"

    IMAGE_FOLDER: str = BASE + "/images_10k"
    IMAGE_TRAIN_PATH: str = IMAGE_FOLDER + "/train"
    IMAGE_VAL_PATH: str = IMAGE_FOLDER + "/val"

    SAMPLE_WEIGHTS_PATH: str = "./data/sample_weights.npy"

## Tensor Declarations

In [ ]:
# Shape aliases that document the tensor layout at each stage of the pipeline.
#
# ``h``/``w`` are the spatial dimensions, ``b`` is the batch size and ``c`` is
# the number of channels/classes.
ImageTensor = Float[Tensor, "3 h w"]  # single image, channel-first layout
IndexMaskTensor = Int64[Tensor, "h w"]
MaskTensor = Float[Tensor, "c h w"]  # one-hot mask, ``c == NUM_CLASSES``
BatchImage = Float[Tensor, "b 3 h w"]  # collated batch of images
BatchIndexMask = Int64[Tensor, "b h w"]
Logits = Float[Tensor, "b c h w"]  # model output, ``c == NUM_CLASSES``
ClassMask = Float[Tensor, "c h w"]  # per-sample probability/binary mask, ``c`` channels
Scalar = Float[Tensor, ""]  # scalar (0-dim) tensor
NumpyImage = Float[np.ndarray, "h w 3"]  # single image, channels-last layout
ClassIndexArray = UInt8[np.ndarray, "h w"]  # per-pixel class id map (numpy)
RawColorImage = UInt8[np.ndarray, "h w 3"]  # single image, channel-last layout

## Class Colors

In [ ]:
# BDD100k color-label palette. The row index is the class id (0-19), matching
# ``Configuration.NUM_CLASSES``. Predicted class maps are coloured with this
# same palette so they render side by side with the ground-truth color labels
# stored on disk. The exact colour per class only needs to be distinct and
# consistent; it mirrors the default BDD100k colours.
CLASS_COLORS: np.ndarray = np.array(
    [
        [128, 64, 128],  # 0  - Road
        [244, 35, 232],  # 1  - Sidewalk
        [70, 70, 70],  # 2  - Building
        [102, 102, 156],  # 3  - Wall
        [190, 153, 153],  # 4  - Fence
        [153, 153, 153],  # 5  - Pole
        [250, 170, 30],  # 6  - Traffic Light
        [220, 220, 0],  # 7  - Traffic Sign
        [107, 142, 35],  # 8  - Vegetation
        [152, 251, 152],  # 9  - Terrain
        [70, 130, 180],  # 10 - Sky
        [220, 20, 60],  # 11 - Person
        [255, 0, 0],  # 12 - Rider
        [0, 0, 142],  # 13 - Car
        [0, 0, 70],  # 14 - Truck
        [0, 60, 100],  # 15 - Bus
        [0, 80, 100],  # 16 - Train
        [0, 0, 230],  # 17 - Motorcycle
        [119, 11, 32],  # 18 - Bicycle
        [0, 0, 0],  # 19 - Unknown
    ],
    dtype=np.uint8,
)

CLASS_NAMES: list[str] = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic light", "traffic sign", "vegetation", "terrain", "sky",
    "person", "rider", "car", "truck", "bus", "train", "motorcycle",
    "bicycle", "unknown",
]


def color_label_to_class_index(label: RawColorImage) -> ClassIndexArray:
    """Map an RGB color-label image to a per-pixel class-index map.

    BDD100k stores segmentation masks as RGB PNGs[cite: 1]. This function converts
    the 3-channel colour values to integer class indices expected by the loss
    function and evaluation metrics. Vectorized 32-bit integer encoding maps
    RGB tuples into single scalar keys to eliminate intermediate array allocations.

    Steps
    -----
    1. Initialize the class index map with the index of the 'unknown' class.
    2. Pack RGB triplets into a single 32-bit integer using bit-shifts.
    3. Pack palette colors using the identical bit-shift formula.
    4. Match packed pixel integers to palette entries and assign class indices.

    Parameters
    ----------
    label : RawColorImage
        RGB color-label array of shape ``(H, W, 3)`` and dtype ``uint8``.

    Returns
    -------
    ClassIndexArray
        Class-index array of shape ``(H, W)`` and dtype ``uint8`` with values in ``[0, 19]``.
    """
    # Initialize all pixels to the fallback Unknown class (index 19)
    class_ids = np.full(label.shape[:2], len(CLASS_COLORS) - 1, dtype=np.uint8)

    # Pack RGB channels into a single int32 scalar: (R << 16) | (G << 8) | B
    # This enables single-operation comparisons without allocating 20 temporary boolean arrays
    label_packed = (
        (label[..., 0].astype(np.int32) << 16)
        | (label[..., 1].astype(np.int32) << 8)
        | label[..., 2].astype(np.int32)
    )
    palette_packed = (
        (CLASS_COLORS[:, 0].astype(np.int32) << 16)
        | (CLASS_COLORS[:, 1].astype(np.int32) << 8)
        | CLASS_COLORS[:, 2].astype(np.int32)
    )

    # Assign class indices where packed pixel keys match palette entries
    for class_id, color_key in enumerate(palette_packed):
        class_ids[label_packed == color_key] = class_id

    return class_ids

## Dataset

In [ ]:
class BDDSegmentationDataset(Dataset[tuple[ImageTensor, MaskTensor]]):
    """Dataset loader for BDD100k semantic segmentation.

    Reads paired camera frames and segmentation maps from disk, applies
    spatial and photometric augmentations, and formats tensors for PyTorch model ingestion.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        transform: A.Compose | None = None,
    ) -> None:
        """Initialize file references and transformation pipeline.

        Parameters
        ----------
        df : pd.DataFrame
            DataFrame containing 'image_paths' and 'mask_paths' columns.
        transform : A.Compose | None, optional
            Albumentations transformation pipeline. Defaults to None.
        """
        super().__init__()
        # Materialize pandas columns to python lists to prevent overhead in __getitem__
        self.image_paths: list[str] = df["image_paths"].to_list()
        self.mask_paths: list[str] = df["mask_paths"].to_list()
        self.transform: A.Compose | None = transform


    def load_sample(self, index: int) -> tuple[NumpyImage, ClassIndexArray]:
        """Load the image and its per-pixel class-index mask at ``index``.

        The image is opened as RGB and normalised to ``[0, 1]``. The mask PNG
        is likewise forced to RGB (some BDD100k color-labels carry an alpha
        channel) before being collapsed from the RGB color-label format to a
        single class id per pixel via :func:`color_label_to_class_index`. This
        class-index representation is what the one-hot encoder and Dice loss
        expect downstream.

        Steps
        -----
        1. Open the image file safely and normalize pixel intensities to [0.0, 1.0].
        2. Open the mask file and translate RGB color encodings into class IDs.
        3. Enforce RGB conversion on both inputs to discard unexpected alpha channels.

        Parameters
        ----------
        index : int
            Dataset index of the sample to load.

        Returns
        -------
        tuple[NumpyImage, ClassIndexArray]
            Tuple of normalized float image ``(H, W, 3)`` and uint8 class index mask ``(H, W)``.

        Raises
        ------
        IndexError
            If ``index`` is out of bounds for the dataset path list.
        """
        if index < 0 or index >= len(self.image_paths):
            raise IndexError(f"Index {index} is out of bounds for dataset of length {len(self)}.")

        image_path = self.image_paths[index]
        mask_path = self.mask_paths[index]

        # Force RGB so that RGBA sources (e.g. some BDD100k color-label PNGs
        # carry an alpha channel) are reduced to 3 channels.
        # Use context managers to guarantee immediate file descriptor release
        with Image.open(image_path) as image_pil:
            image = np.array(image_pil.convert("RGB"), dtype=np.float32) / 255.0

        with Image.open(mask_path) as mask_pil:
            raw_mask = np.array(mask_pil.convert("RGB"), dtype=np.uint8)
            class_mask = color_label_to_class_index(raw_mask)

        return image, class_mask


    def __len__(self) -> int:
        """Return the total number of samples present in the dataset.

        Returns
        -------
        int
            Length of the underlying sample list.
        """
        return len(self.image_paths)


    def __getitem__(self, index: int) -> tuple[ImageTensor, IndexMaskTensor]:
        """Retrieve and transform the sample at position ``index``.

        Steps
        -----
        1. Fetch raw image array and 2D integer class mask via ``load_sample``.
        2. Apply spatial and color augmentations via Albumentations.
        3. Cast transformed mask to int64 without one-hot expansion to conserve memory.

        Parameters
        ----------
        index : int
            Index of the sample to retrieve.

        Returns
        -------
        tuple[ImageTensor, IndexMaskTensor]
            Transformed image tensor ``(3, H, W)`` and integer mask tensor ``(H, W)``.
        """
        image, class_mask = self.load_sample(index)

        # Apply Albumentations pipeline or default channel transposition
        if self.transform:
            transformed = self.transform(image=image, mask=class_mask)
        else:
            transformed = ToTensorV2()(image=image, mask=class_mask)

        # Retain the mask as a compact 2D int64 tensor (H, W).
        # Expanding to a 20-channel one-hot float tensor would inflate host memory
        # by 20x and cause kernel OOM crashes during epoch collation.
        mask_tensor = transformed["mask"].to(torch.int64)

        return transformed["image"], mask_tensor

## Utility Functions

In [ ]:
def find_image_path_from_mask(complete_mask_path: str, base_image_path: str) -> str:
    file_path_split = complete_mask_path.split("/")
    mask_file_name = file_path_split[-1].split("_")[0]

    image_path = base_image_path + "/" + mask_file_name + ".jpg"
    return image_path


def find_train_image_path_from_mask(complete_mask_path: str) -> str:
    return find_image_path_from_mask(complete_mask_path, Path.IMAGE_TRAIN_PATH)


def find_val_image_path_from_mask(complete_mask_path: str) -> str:
    return find_image_path_from_mask(complete_mask_path, Path.IMAGE_VAL_PATH)


def find_mask_path_from_image(complete_image_path: str, base_mask_path: str) -> str:
    file_path_split = complete_image_path.split("/")
    mask_file_name = file_path_split[-1].split(".")[0]

    mask_path = base_mask_path + "/" + mask_file_name + "_train_color.png"
    return mask_path


def find_train_mask_path_from_image(complete_image_path: str) -> str:
    return find_mask_path_from_image(complete_image_path, Path.SEGMENTATION_MASK_TRAIN_PATH)


def find_val_mask_path_from_image(complete_image_path: str) -> str:
    return find_mask_path_from_image(complete_image_path, Path.SEGMENTATION_MASK_VAL_PATH)


def load_dataset_from_files() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # load train, then split into train-test
    train_mask_paths = glob.glob(f"{Path.SEGMENTATION_MASK_TRAIN_PATH}/*.png")
    problematic_masks = []
    for complete_mask_path in train_mask_paths:
        with Image.open(complete_mask_path) as img:
            width, height = img.size
            if width != 1280 or height != 720:
                problematic_masks.append(complete_mask_path)
                train_mask_paths.remove(complete_mask_path)
    print(f"Problematic masks: {problematic_masks}")

    train_image_paths = list(map(find_train_image_path_from_mask, train_mask_paths))
    problematic_images = []
    for complete_image_path in train_image_paths:
        with Image.open(complete_image_path) as img:
            width, height = img.size
            if width != 1280 or height != 720:
                problematic_images.append(complete_image_path)
                train_image_paths.remove(complete_image_path)
                train_mask_paths.remove(find_train_mask_path_from_image(complete_image_path))
    print(f"Problematic images: {problematic_images}")

    train_test_df = pd.DataFrame({
        "image_paths": train_image_paths,
        "mask_paths": train_mask_paths,
    })
    train_df, test_df = train_test_split(train_test_df, test_size=0.2, random_state=Configuration.SEED)

    # load val
    val_mask_paths = glob.glob(f"{Path.SEGMENTATION_MASK_VAL_PATH}/*.png")
    problematic_val_masks = []
    for complete_mask_path in val_mask_paths:
        with Image.open(complete_mask_path) as img:
            width, height = img.size
            if width != 1280 or height != 720:
                problematic_val_masks.append(complete_mask_path)
                val_mask_paths.remove(complete_mask_path)
    print(f"Problematic val masks: {problematic_val_masks}")

    val_image_paths = list(map(find_val_image_path_from_mask, val_mask_paths))
    problematic_val_images = []
    for complete_image_path in val_image_paths:
        with Image.open(complete_image_path) as img:
            width, height = img.size
            if width != 1280 or height != 720:
                problematic_val_images.append(complete_image_path)
                val_image_paths.remove(complete_image_path)
                val_mask_paths.remove(find_val_mask_path_from_image(complete_image_path))
    print(f"Problematic val images: {problematic_val_images}")

    val_df = pd.DataFrame({
        "image_paths": val_image_paths,
        "mask_paths": val_mask_paths,
    })

    return train_df, val_df, test_df


def forward(model: nn.Module, x: BatchImage) -> Logits:
    """Execute a forward pass through the model and cast the result to Logits.

    Steps
    -----
    1. Run input tensor batch through the neural network.
    2. Return logits for downstream loss computation and type verification.

    Parameters
    ----------
    model : nn.Module
        Segmentation network mapping ``(B, 3, H, W)`` to ``(B, C, H, W)``[cite: 1].
    x : BatchImage
        Batch of input images of shape ``(B, 3, H, W)``.

    Returns
    -------
    Logits
        Raw output logits tensor of shape ``(B, NUM_CLASSES, H, W)``.
    """
    return model(x)


def kl_divergence(probs: Logits, adv_probs: Logits, eps: float = 1e-8) -> Scalar:
    """Compute mean per-pixel Kullback-Leibler divergence between predictions.

    TRADES measures adversarial perturbation impact by calculating the divergence
    between clean predictions and perturbed predictions[cite: 1]. Computing this per-pixel
    and averaging over batch and spatial dimensions keeps the regularizer scale
    invariant to input resolution.

    Steps
    -----
    1. Add ``eps`` to both probability distributions to prevent ``log(0)``.
    2. Sum ``probs * (log(probs) - log(adv_probs))`` over the class channel.
    3. Compute spatial and batch mean to yield a scalar divergence value.

    Parameters
    ----------
    probs : Logits
        Softmax probability tensor of clean predictions ``(B, C, H, W)``.
    adv_probs : Logits
        Softmax probability tensor of adversarial predictions ``(B, C, H, W)``.
    eps : float, optional
        Numerical stability epsilon to prevent undefined log evaluations. Defaults to 1e-8.

    Returns
    -------
    Scalar
        Averaged scalar KL-divergence tensor.
    """
    # Prevent undefined log values when probabilities are confident zeros
    kl_per_pixel = (probs * (torch.log(probs + eps) - torch.log(adv_probs + eps))).sum(dim=1)
    return kl_per_pixel.mean()

## Loss Functions

In [ ]:
class CompoundLoss(nn.Module):
    """Compound loss combining Dice Loss and Focal Loss for semantic segmentation.

    Semantic segmentation on class-imbalanced datasets benefits from combining a
    region-based loss (Dice loss) and a distribution-based loss (Focal loss).
    Dice loss optimizes overall mask overlap to handle class imbalance, while
    Focal loss down-weights easy background pixels to focus gradient updates on
    hard boundaries and minority classes.

    Steps
    -----
    1. Initialize the underlying SMP DiceLoss and FocalLoss modules in multiclass mode.
    2. In the forward pass, inspect target tensor dimensionality; if 4D (one-hot encoded),
       collapse the channel dimension to 2D class indices via ``argmax(dim=1)``.
    3. Compute multi-class Dice loss from raw logits.
    4. Compute multi-class Focal loss from raw logits.
    5. Return the weighted sum: ``dice_weight * dice + focal_weight * focal``.

    Parameters
    ----------
    dice_weight : float, optional
        Weight factor applied to the Dice loss component. Defaults to 0.5.
    focal_weight : float, optional
        Weight factor applied to the Focal loss component. Defaults to 1.0.
    ignore_index : int, optional
        Class index to ignore during loss computation. Defaults to 19.
    """

    def __init__(
        self,
        dice_weight: float = 0.5,
        focal_weight: float = 1.0,
        ignore_index: int = 19,
    ) -> None:
        super().__init__()
        # Store loss weighting factors to adjust the relative contribution of each loss term
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight

        # SMP DiceLoss operates directly on raw logits when from_logits=True,
        # avoiding an explicit external softmax step
        self.dice_loss = smp.losses.DiceLoss(
            mode=smp.losses.MULTICLASS_MODE,
            from_logits=True,
            ignore_index=ignore_index,
        )

        # SMP FocalLoss handles multiclass cross-entropy while masking out
        # the designated unknown/ignored class id
        self.focal_loss = smp.losses.FocalLoss(
            mode=smp.losses.MULTICLASS_MODE,
            ignore_index=ignore_index,
        )

    def forward(self, logits: Tensor, targets: Tensor) -> Tensor:
        """Compute the weighted compound loss between predictions and targets.

        Steps
        -----
        1. Check if targets are 4D (one-hot encoded); if so, collapse to class indices.
        2. Evaluate Dice loss and Focal loss independently.
        3. Weight and sum both loss terms.

        Parameters
        ----------
        logits : Tensor
            Raw unnormalized model predictions of shape ``(B, C, H, W)``.
        targets : Tensor
            Ground-truth masks, either one-hot encoded tensors of shape ``(B, C, H, W)``
            or class indices of shape ``(B, H, W)``.

        Returns
        -------
        Tensor
            Scalar compound loss tensor suitable for gradient backpropagation.
        """
        # Collapse one-hot targets (B, C, H, W) to class indices (B, H, W) because
        # SMP multiclass loss functions with ignore_index require integer class indices
        if targets.ndim == 4:
            targets = targets.argmax(dim=1)

        dice = cast(Scalar, self.dice_loss(logits, targets))
        focal = cast(Scalar, self.focal_loss(logits, targets))

        # Combine weighted losses to balance boundary refinement and region overlap
        return self.dice_weight * dice + self.focal_weight * focal


class TradesLoss(nn.Module):
    """TRADES adversarial loss for semantic segmentation.

    Combines a natural (clean) loss with a KL-divergence robustness term, as
    proposed by Zhang et al. (2019) "Theoretically Principled Trade-off between
    Robustness and Accuracy". The natural loss encourages accurate clean
    predictions, while the robustness term pushes the model to keep its output
    stable for adversarial inputs inside an ``epsilon`` ball around each sample.

    The adversarial input is crafted with projected gradient descent (PGD): for
    ``num_steps`` iterations the perturbation is moved by ``alpha`` in the
    direction of the signed gradient of ``KL(clean || adversarial)`` and then
    projected back onto the ``L_inf`` ball of radius ``epsilon``. The final
    loss is ``natural_loss(clean) + beta * KL(clean || adversarial)``.

    Parameters
    ----------
    natural_loss : nn.Module
        Loss applied to clean logits, e.g. :class:`DiceLoss`.
    epsilon : float, optional
        Radius of the ``L_inf`` adversarial perturbation ball. Defaults to 0.03.
    alpha : float, optional
        PGD step size. Defaults to 0.01.
    num_steps : int, optional
        Number of PGD iterations used to craft the perturbation. Defaults to 10.
    beta : float, optional
        Weight of the robustness term relative to the natural loss. Defaults to
        6.0.

    Notes
    -----
    The ``model`` is passed to :meth:`forward` rather than stored on the
    instance, so the loss object stays independent of any particular network
    checkpoint and can be reused across runs.
    """

    def __init__(
        self,
        natural_loss: nn.Module,
        epsilon: float = 0.03,
        alpha: float = 0.01,
        num_steps: int = 10,
        beta: float = 6.0,
    ) -> None:
        """Configure TRADES perturbation boundaries and weighting hyperparameters.

        Parameters
        ----------
        natural_loss : nn.Module
            Supervised criterion applied to clean model logits.
        epsilon : float, optional
            L-infinity perturbation boundary radius. Defaults to 0.03.
        alpha : float, optional
            Single step perturbation increment. Defaults to 0.01.
        num_steps : int, optional
            Number of PGD iterations per batch. Defaults to 10.
        beta : float, optional
            Trade-off weight balancing natural loss against robustness. Defaults to 6.0.
        """

        super().__init__()
        self.natural_loss: nn.Module = natural_loss
        self.epsilon: float = epsilon
        self.alpha: float = alpha
        self.num_steps: int = num_steps
        self.beta: float = beta

    def forward(
        self,
        model: nn.Module,
        x: BatchImage,
        y: BatchIndexMask,
    ) -> tuple[Scalar, Logits]:
        """Compute natural loss, craft PGD adversary, and evaluate robustness loss.

        Steps
        -----
        1. Evaluate clean forward pass, compute natural loss, and detach clean probabilities.
        2. Set model to eval mode during PGD so BatchNorm running statistics are not
           polluted by intermediate adversarial steps.
        3. Iteratively ascend the KL divergence gradient bounded by the epsilon ball.
        4. Restore model training state and evaluate the final robustness regularizer.
        5. Return the total loss and the detached clean logits for metric tracking.

        Parameters
        ----------
        model : nn.Module
            Segmentation network being optimized.
        x : BatchImage
            Clean image batch of shape ``(B, 3, H, W)``.
        y : BatchIndexMask
            Target class index batch of shape ``(B, H, W)``.

        Returns
        -------
        tuple[Scalar, Logits]
            Tuple containing the differentiable composite loss and detached clean logits.
        """
        # 1. Clean branch: computes baseline predictions and task loss
        clean_logits = forward(model, x)
        natural = self.natural_loss(clean_logits, y)

        # Detach clean probabilities so gradient updates only affect the perturbed branch
        clean_probs = torch.softmax(clean_logits, dim=1).detach()

        # 2. PGD Attack: Freeze BatchNorm updates while crafting perturbations.
        # Running BatchNorm in training mode over 10 PGD iterations corrupts moving
        # mean/variance statistics and creates unnecessary graph retention.
        was_training = model.training
        model.eval()

        x_adv = x.clone().detach()
        for _ in range(self.num_steps):
            # Re-enable autograd on the current adversarial image so the step
            # walks the loss surface of the robustness term.
            x_adv.requires_grad_(True)

            adv_logits = forward(model, x_adv)
            adv_probs = torch.softmax(adv_logits, dim=1)
            kl = kl_divergence(clean_probs, adv_probs)

            # Calculate gradient of divergence with respect to the input
            grad = torch.autograd.grad(kl, x_adv)[0]

            # Step along sign of gradient and project back onto epsilon ball and [0, 1] bounds
            perturbation = x_adv.detach() + self.alpha * grad.sign() - x
            perturbation = torch.clamp(perturbation, -self.epsilon, self.epsilon)
            x_adv = torch.clamp(x + perturbation, 0.0, 1.0).detach()

        # Restore original training state before final forward pass
        if was_training:
            model.train()

        # 3. Robustness regularizer on final adversarial example
        # Robustness term: penalise how much the final adversarial input moved
        # the model's prediction away from the clean prediction.
        adv_logits = forward(model, x_adv)
        adv_probs = torch.softmax(adv_logits, dim=1)
        robust = kl_divergence(clean_probs, adv_probs)

        total_loss = natural + self.beta * robust

        # Return detached clean logits alongside total loss to avoid redundant forward passes
        return total_loss, clean_logits.detach()


## Training and Evaluation

In [ ]:
@torch.no_grad()
def compute_batch_macro_iou(
    y_pred: torch.Tensor,      # (B, C, H, W) logits
    y_true: torch.Tensor,      # (B, C, H, W) one-hot
    num_classes: int = 19,     # Classes 0 to 18 (excludes 19: Unknown)
    eps: float = 1e-7,
) -> float:
    """Compute the mean macro IoU over a batch, excluding ignored classes.

    The model outputs raw logits while the ground truth is one-hot, so both are
    first reduced to a single class id per pixel via ``argmax``. IoU is then
    computed class by class and averaged only over the classes that actually
    occur in either the prediction or the ground truth; empty classes are
    skipped rather than counted as zero, which would otherwise drag the mean
    down on batches dominated by a few classes.

    Steps
    -----
    1. Extract class predictions by finding argmax over the logits channel axis.
    2. For each class ID in ``[0, num_classes)``, calculate intersection and union masks.
    3. Exclude classes that appear in neither ground-truth nor prediction to avoid skewing the average.
    4. Compute mean IoU across all present classes.

    Parameters
    ----------
    y_pred : Logits
        Predicted logits tensor of shape ``(B, C, H, W)``.
    y_true : BatchIndexMask
        Ground-truth class indices of shape ``(B, H, W)``.
    num_classes : int, optional
        Number of classes evaluated, excluding background/unknown. Defaults to 19.
    eps : float, optional
        Division smoothing constant to prevent zero-division errors. Defaults to 1e-7.

    Returns
    -------
    float
        Macro IoU score averaged across active classes in the batch.
    """
    preds = y_pred.argmax(dim=1)

    iou_per_class = []
    for cls in range(num_classes):
        pred_mask = preds == cls
        true_mask = y_true == cls

        intersection = (pred_mask & true_mask).sum().float().item()
        union = (pred_mask | true_mask).sum().float().item()

        # Only evaluate classes that actually occur in either prediction or ground truth
        if union > 0:
            iou_per_class.append((intersection + eps) / (union + eps))

    return float(np.mean(iou_per_class)) if iou_per_class else 0.0


def execute_epoch(
    model: nn.Module,
    dataloader: DataLoader[tuple[ImageTensor, IndexMaskTensor]],
    optimizer: torch.optim.Optimizer,
    loss_fn: TradesLoss,
    device: torch.device,
) -> tuple[float, float]:
    """Execute one training epoch using TRADES adversarial loss.

    Steps
    -----
    1. Set the model to train mode and initialize metric accumulators.
    2. Iterate over dataloader batches, transferring tensors to the designated device.
    3. Zero optimiser gradients, evaluate TRADES loss, backpropagate, and step weights.
    4. Accumulate batch loss and macro IoU using detached clean logits from the loss call.
    5. Return epoch-level average loss and IoU.

    Parameters
    ----------
    model : nn.Module
        Segmentation model being trained.
    dataloader : DataLoader[tuple[ImageTensor, IndexMaskTensor]]
        Batched training data loader.
    optimizer : torch.optim.Optimizer
        Optimization algorithm managing model weights.
    loss_fn : TradesLoss
        TRADES objective function generating adversarial perturbations.
    device : torch.device
        Hardware accelerator running the computation.

    Returns
    -------
    tuple[float, float]
        Tuple of (mean_epoch_loss, mean_epoch_macro_iou).
    """

    # Set model into training mode
    model.train()

    # Initialize train loss & accuracy
    train_loss, train_iou = 0.0, 0.0

    # Execute training loop over train dataloader
    for X, y in dataloader:
        # Transfer batch to target device asynchronously with non_blocking=True
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        # Reset Gradient
        optimizer.zero_grad()

        # TradesLoss computes loss and returns detached clean logits in a single unified step
        loss, clean_logits = loss_fn(model, X, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        # Compute IoU from the detached clean logits without retaining an un-backwarded graph
        train_iou += compute_batch_macro_iou(clean_logits, y, num_classes=19)


    # Compute Step Metrics
    train_loss = train_loss / len(dataloader)
    train_iou = train_iou / len(dataloader)

    return train_loss, train_iou


def evaluate(
    model: nn.Module,
    dataloader: DataLoader[tuple[ImageTensor, IndexMaskTensor]],
    loss_fn: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    """Evaluate clean performance metrics on validation data.

    Steps
    -----
    1. Set model to eval mode and enter ``torch.inference_mode`` context.
    2. Pass clean validation batches through the model without gradient tracking.
    3. Accumulate validation loss and macro IoU scores.
    4. Return normalized validation metrics.

    Parameters
    ----------
    model : nn.Module
        Segmentation model under evaluation.
    dataloader : DataLoader[tuple[ImageTensor, IndexMaskTensor]]
        Validation data loader.
    loss_fn : nn.Module
        Standard clean loss criterion.
    device : torch.device
        Hardware accelerator running evaluation.

    Returns
    -------
    tuple[float, float]
        Tuple of (mean_eval_loss, mean_eval_macro_iou).
    """
    model.eval()
    eval_loss, eval_iou = 0.0, 0.0

    # inference_mode provides greater memory savings and speedups over no_grad
    with torch.inference_mode():
        for X, y in dataloader:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            y_pred = forward(model, X)
            loss = loss_fn(y_pred, y)

            eval_loss += loss.item()
            eval_iou += compute_batch_macro_iou(y_pred, y, num_classes=19)

    eval_loss = eval_loss / len(dataloader)
    eval_iou = eval_iou / len(dataloader)

    return eval_loss, eval_iou


def train(
    model: nn.Module,
    train_dataloader: DataLoader[tuple[ImageTensor, IndexMaskTensor]],
    eval_dataloader: DataLoader[tuple[ImageTensor, IndexMaskTensor]],
    optimizer: torch.optim.Optimizer,
    loss_fn: TradesLoss,
    eval_loss_fn: nn.Module,
    epochs: int,
    train_device: torch.device,
    eval_device: torch.device,
    scheduler: lr_scheduler.ReduceLROnPlateau | None = None,
) -> tuple[nn.Module, Dict[str, List[float]]]:
    """Orchestrate training loop, validation tracking, checkpoints, and memory hygiene.

    Steps
    -----
    1. Initialize training history dictionary and checkpoint tracking variables.
    2. For each epoch, execute training with ``TradesLoss`` and validation with clean loss.
    3. Save an un-wrapped CPU state dict copy whenever validation loss reaches a new minimum.
    4. Step learning rate scheduler if provided.
    5. Call ``gc.collect()`` and ``torch.cuda.empty_cache()`` at epoch end to prevent memory fragmentation.
    6. Revert model parameters to best validation checkpoint before returning.

    Parameters
    ----------
    model : nn.Module
        Segmentation model to train.
    train_dataloader : DataLoader[tuple[ImageTensor, IndexMaskTensor]]
        Batched training data loader.
    eval_dataloader : DataLoader[tuple[ImageTensor, IndexMaskTensor]]
        Batched validation data loader.
    optimizer : torch.optim.Optimizer
        Model parameter optimizer.
    loss_fn : TradesLoss
        Adversarial TRADES objective.
    eval_loss_fn : nn.Module
        Clean validation loss objective.
    epochs : int
        Number of complete training epochs.
    train_device : torch.device
        Hardware accelerator used for training.
    eval_device : torch.device
        Hardware accelerator used for validation.
    scheduler : lr_scheduler.ReduceLROnPlateau | None, optional
        Learning rate scheduler stepped on validation loss. Defaults to None.

    Returns
    -------
    tuple[nn.Module, Dict[str, List[float]]]
        Trained model restored to best validation weights and dictionary of history metrics.
    """
    session: Dict[str, List[float]] = {
        "loss": [],
        "macro_iou_score": [],
        "eval_loss": [],
        "eval_macro_iou_score": [],
    }

    best_eval_loss = float("inf")
    best_model_state: Dict[str, Tensor] | None = None

    for epoch in tqdm(range(epochs)):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        train_loss, train_iou = execute_epoch(
            model=model,
            dataloader=train_dataloader,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=train_device,
        )

        eval_loss, eval_iou = evaluate(
            model=model,
            dataloader=eval_dataloader,
            loss_fn=eval_loss_fn,
            device=eval_device,
        )

        # Access underlying module when wrapped inside DataParallel to ensure clean checkpoints
        raw_model = model.module if hasattr(model, "module") else model

        # Preserve model state whenever validation loss improves
        if eval_loss < best_eval_loss:
            best_eval_loss = eval_loss
            best_model_state = {
                name: param.detach().cpu().clone()
                for name, param in raw_model.state_dict().items()
            }

        # Step learning rate scheduler if present
        current_lr = optimizer.param_groups[0]["lr"]
        if scheduler:
            scheduler.step(eval_loss)
            current_lr = optimizer.param_groups[0]["lr"]

        log_text = (
            f"loss: {train_loss:.4f} - train_macro_iou: {train_iou:.4f} - "
            f"eval_loss: {eval_loss:.4f} - eval_macro_iou_score: {eval_iou:.4f}"
        )
        if scheduler:
            print(log_text + f" - lr: {current_lr}")
        else:
            print(log_text)

        session["loss"].append(train_loss)
        session["macro_iou_score"].append(train_iou)
        session["eval_loss"].append(eval_loss)
        session["eval_macro_iou_score"].append(eval_iou)

        # Explicitly invoke Python garbage collection and release cached PyTorch memory.
        # This prevents fragmented tensors from steadily accumulating in host memory across epochs.
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Restore the model to its best recorded validation weights
    if best_model_state is not None:
        raw_model = model.module if hasattr(model, "module") else model
        raw_model.load_state_dict(best_model_state)

    return raw_model, session


def plot_training_curves(
    history: Dict[str, List[float]],
    fig_size: tuple[int, int] = (20, 10)
) -> None:

    loss = np.array(history['loss'])
    val_loss = np.array(history['eval_loss'])

    iou = np.array(history['macro_iou_score'])
    val_iou = np.array(history['eval_macro_iou_score'])

    epochs = range(len(history['loss']))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=fig_size)

    # Plot loss
    ax1.plot(epochs, loss, label='training_loss', marker='o', color='C5')
    ax1.plot(epochs, val_loss, label='eval_loss', marker='o', color='C6')

    # Fill area between losses
    ax1.fill_between(epochs, loss, val_loss, where=(loss > val_loss), color='C5', alpha=0.4, interpolate=True)
    ax1.fill_between(epochs, loss, val_loss, where=(loss < val_loss), color='C6', alpha=0.4, interpolate=True)

    # Add Text & Formats
    ax1.set_title('Loss (Lower Means Better)', fontsize=22)
    ax1.set_xlabel('Epochs', fontsize=18)
    ax1.set_ylabel('Loss', fontsize=18)
    ax1.tick_params(axis='both', which='major', labelsize=14)
    ax1.legend(fontsize=14)

    # Plot metric
    ax2.plot(epochs, iou, label='training_macro_iou', marker='o', color='C5')
    ax2.plot(epochs, val_iou, label='eval_macro_iou', marker='o', color='C6')

    # Fill area between metrics
    ax2.fill_between(epochs, iou, val_iou, where=(iou > val_iou), color='C5', alpha=0.4, interpolate=True)
    ax2.fill_between(epochs, iou, val_iou, where=(iou < val_iou), color='C6', alpha=0.4, interpolate=True)

    # Add Text & Formats
    ax2.set_title('Macro IoU (Higher Means Better)', fontsize=22)
    ax2.set_xlabel('Epochs', fontsize=18)
    ax2.set_ylabel('Macro IoU', fontsize=18)
    ax2.tick_params(axis='both', which='major', labelsize=14)
    ax2.legend(fontsize=14)
    sns.despine()


def colorize_mask(class_mask: np.ndarray, palette: np.ndarray) -> np.ndarray:
    """Map a class-index mask to an RGB image using ``palette``.

    Steps
    -----
    1. Cast ``class_mask`` to integer so it can be used as row indices.
    2. Index ``palette`` with those indices, turning a ``(H, W)`` array of
       class ids into a ``(H, W, 3)`` image.

    Parameters
    ----------
    class_mask : np.ndarray
        Array of shape ``(H, W)`` whose values are class indices.
    palette : np.ndarray
        Array of shape ``(NUM_CLASSES, 3)`` mapping a class id to an RGB
        colour (the 0-255 range).

    Returns
    -------
    np.ndarray
        RGB image of shape ``(H, W, 3)`` matching the dtype of ``palette``.
    """
    return palette[class_mask.astype(np.int64)]


def visualize_predictions(
    model: nn.Module,
    test_df: pd.DataFrame,
    device: torch.device,
    num_samples: int = 4,
    output_path: str = "./model/predictions.png",
) -> None:
    """Render test samples side by side with ground-truth and predicted segmentation masks.

    Steps
    -----
    1. Select the first ``num_samples`` rows from the evaluation DataFrame.
    2. Pass each image through the model in inference mode to generate logits.
    3. Colorize ground-truth and predicted masks with the BDD100k color palette.
    4. Export comparison subplots to a PNG file and close the figure to free RAM.

    Parameters
    ----------
    model : nn.Module
        Trained segmentation model.
    test_df : pd.DataFrame
        DataFrame containing sample image and mask paths.
    device : torch.device
        Hardware accelerator running inference.
    num_samples : int, optional
        Number of test images to render. Defaults to 4.
    output_path : str, optional
        Filesystem destination for exported comparison PNG. Defaults to "./model/predictions.png".

    Raises
    ------
    ValueError
        If ``test_df`` is empty.
    """
    sample_df = test_df.head(min(num_samples, len(test_df))).reset_index(drop=True)
    if sample_df.empty:
        raise ValueError("test_df has no rows to visualize.")

    num_rows = len(sample_df)
    fig, axes = plt.subplots(num_rows, 3, figsize=(15, 5 * num_rows))
    if num_rows == 1:
        axes = axes[np.newaxis, :]

    sample_ds = BDDSegmentationDataset(sample_df)
    eval_model = model.module if hasattr(model, "module") else model
    eval_model.eval()

    for row in range(num_rows):
        image, class_mask = sample_ds.load_sample(row)

        # Prepare tensor for single-sample inference (1, 3, H, W)
        image_tensor = torch.from_numpy(image.transpose(2, 0, 1)).contiguous()
        image_tensor = image_tensor.unsqueeze(0).to(device)

        with torch.inference_mode():
            logits = eval_model(image_tensor)

        pred_class = logits.argmax(dim=1).squeeze(0).cpu().numpy()

        pred_color = colorize_mask(pred_class, CLASS_COLORS).astype(np.float32) / 255.0
        true_mask_color = colorize_mask(class_mask, CLASS_COLORS).astype(np.float32) / 255.0

        axes[row, 0].imshow(image)
        axes[row, 0].set_title("Image")

        axes[row, 1].imshow(image)
        axes[row, 1].imshow(true_mask_color, alpha=0.5)
        axes[row, 1].set_title("Image + True Mask")

        axes[row, 2].imshow(image)
        axes[row, 2].imshow(pred_color, alpha=0.5)
        axes[row, 2].set_title("Image + Predicted Mask")

        for ax in axes[row]:
            ax.set_xticks([])
            ax.set_yticks([])

    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


## Main

In [ ]:
# Print current Torch package versions
print('Package versions:')
print('*'*26)
print(f'torch \t\t - {torch.__version__}')
print(f'torchvision \t - {torchvision.__version__}')

train_df, val_df, test_df = load_dataset_from_files()

train_transforms = A.Compose([
    A.Resize(height=Configuration.IMAGE_HEIGHT, width=Configuration.IMAGE_WIDTH),
    A.RandomBrightnessContrast(p=0.2),
    A.HorizontalFlip(p=0.5),
    # The mask is now a 2-D class-index map, so ``ToTensorV2`` needs no
    # ``transpose_mask``: it leaves the (H, W) mask as-is and only converts
    # the image to (C, H, W).
    ToTensorV2(),
])

inference_transforms = A.Compose([
    A.Resize(height=Configuration.IMAGE_HEIGHT, width=Configuration.IMAGE_WIDTH),
    ToTensorV2(),
])
train_ds = BDDSegmentationDataset(train_df, transform=train_transforms)
val_ds = BDDSegmentationDataset(val_df, transform=inference_transforms)

# Always assume that sample weights are available
print(f"Reusing precomputed sample weights from '{Path.SAMPLE_WEIGHTS_PATH}'...")
train_sample_weights = np.load(Path.SAMPLE_WEIGHTS_PATH)

train_sampler = WeightedRandomSampler(
    weights=train_sample_weights.tolist(),
    num_samples=len(train_ds),
    replacement=True,
)

# DataLoader configuration:
# 1. num_workers=Configuration.NUM_WORKERS: loads batches asynchronously across background processes[cite: 1].
# 2. pin_memory=True: allocates page-locked host memory, accelerating host-to-device transfers over PCIe.
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=Configuration.BATCH_SIZE,
    sampler=train_sampler,
    num_workers=Configuration.NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    dataset=val_ds,
    batch_size=Configuration.BATCH_SIZE,
    shuffle=Configuration.APPLY_SHUFFLE,
    num_workers=Configuration.NUM_WORKERS,
    pin_memory=True,
)

model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=Configuration.CHANNELS,
    classes=Configuration.NUM_CLASSES,
)
model = model.to(Configuration.DEVICE)

print(
    summary(
            model=model,
            input_size=(Configuration.BATCH_SIZE, Configuration.CHANNELS, Configuration.IMAGE_HEIGHT, Configuration.IMAGE_WIDTH),
            col_names=["output_size", "num_params", "trainable"],
            col_width=30,
            row_settings=["var_names"],
            depth=5
        )
)

# Wrap model in DataParallel if 2 or more GPUs are present
if torch.cuda.device_count() > 1:
    print(f"Utilizing {torch.cuda.device_count()} GPUs with DataParallel!")
    model = nn.DataParallel(model)

# Define the natural loss. An :class:`CompoundLoss` ``nn.Module`` is used
# rather than a bare function so it satisfies the ``natural_loss: nn.Module``
# annotation on :class:`TradesLoss`. It also collapses the one-hot masks it
# receives from the TRADES loss back into class indices automatically.
natural_loss = CompoundLoss(
    dice_weight=0.5,
    focal_weight=1.0,
    ignore_index=19,
)

loss_fn = TradesLoss(natural_loss=natural_loss)

# Define optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=Configuration.LR,
)

# Define Scheduler
scheduler = lr_scheduler.ReduceLROnPlateau(
    optimizer=optimizer,
    mode="min",
    patience=Configuration.PATIENCE,
)

print('Training U-Net Model')
print(f'Train on {len(train_df)} samples, validate on {len(val_df)} samples.')
print('----------------------------------')

# Generate training session config
session_config = {
    'model'               : model,
    'train_dataloader'    : train_loader,
    'eval_dataloader'     : val_loader,
    'optimizer'           : optimizer,
    'scheduler'           : scheduler,
    'loss_fn'             : loss_fn,
    'eval_loss_fn'        : loss_fn.natural_loss,
    'epochs'              : Configuration.EPOCHS,
    'train_device'        : Configuration.DEVICE,
    'eval_device'         : Configuration.DEVICE,
}

# Execute Training Session
model, unet_session_history = train(**session_config)

# Create Model directory
model_name = 'teacher'
model_path = './model/'
os.makedirs(model_path, exist_ok=True)

# Save Model
torch.save(model, os.path.join(model_path, f"{model_name}.pt"))

# Convert U-Net history dict to DataFrame
unet_session_history_df = pd.DataFrame(unet_session_history)
print(unet_session_history_df)

# Plot U-Net Session Training History
plot_training_curves(
    unet_session_history,
    fig_size=(20, 20)
)

# Export a grid of test samples (image / image+true mask /
# image+predicted mask) so the model output can be inspected visually.
visualize_predictions(
    model,
    test_df,
    torch.device(Configuration.DEVICE)
)
